# 3 · Memory and rhythm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/EN/jupyter/markov-book-en/03_memory_and_rhythm.ipynb)

Two questions for this chapter:

1. How much **memory** should the chain have?
2. How do we teach it **rhythm**, not just pitch?

In [ ]:
# Setup -- runs by itself (about 20 seconds the first time)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # which version of GreasyPidgin to use (a git branch or tag)
TUTORIAL_REF  = "main"   # which version of this tutorial's helper file to use
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/EN/jupyter/markov-book-en/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

print("Ready!")

In [ ]:
midi_path = wh.choose_midi()
melody = wh.load_melody(midi_path)
print(melody)

## Memory: copying versus inventing

For each order we learn a chain, generate a melody and measure how much of it is a **verbatim copy**
of the original. `copy_ratio` is the share of 4-note groups in the new melody that also occur,
unchanged, in the original. (`random.seed` makes the experiment repeatable.)

In [ ]:
results = {}

for order in (1, 2, 3, 4):
    chain = MarkovChain()
    chain.generateRulesFromList(melody.pitches, order=order)

    random.seed(3)
    seq = chain.getNElements(seed=tuple(melody.pitches[:order]), n=48)
    results[order] = seq

    ratio = wh.copy_ratio(melody.pitches, seq, n=4)
    print(f"order {order}:  {len(chain.rules):3d} rules   copied: {ratio:4.0%}")

With more memory the chain has more rules, but each rule has fewer options. At some point it is
just replaying the original. **Low order: surprising, often incoherent. High order: coherent, but a copy.**

Listen to the extremes:

In [ ]:
for order in (1, 4):
    print(f"order {order}")
    wh.render_midi(results[order], melody.deltas, melody.bpm, f"order{order}.mid")
    wh.listen(f"order{order}.mid")

### The rules as a picture

Orders 1 and 2 side by side. In order 2 every row is a *pair* of notes.
Rows with a single bright cell are places where the chain has no choice.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, order in zip(axes, (1, 2)):
    chain = MarkovChain()
    chain.generateRulesFromList(melody.pitches, order=order)
    wh.plot_transition_matrix(chain, ax=ax)
    ax.set_title(f"order {order}")
plt.tight_layout()
plt.show()

## Rhythm: a second chain

Rhythm is just another sequence of symbols: the time between note starts, in beats
(`melody.deltas`). So we learn a **second chain** for it, exactly like the first one.

In [ ]:
rhythm_chain = MarkovChain()
rhythm_chain.generateRulesFromList(melody.deltas, order=2)

pitch_chain = MarkovChain()
pitch_chain.generateRulesFromList(melody.pitches, order=2)

N = 32
random.seed(None)
new_pitches = pitch_chain.getNElements(seed=tuple(melody.pitches[:2]), n=N)
new_rhythm  = rhythm_chain.getNElements(seed=tuple(melody.deltas[:2]), n=N)

print("pitches:", " ".join(wh.pitch_name(p) for p in new_pitches[:12]), "...")
print("rhythm: ", new_rhythm[:12], "...")

In [ ]:
wh.render_midi(new_pitches, new_rhythm, melody.bpm, "separate_chains.mid")
wh.plot_notes(new_pitches, new_rhythm, title="Two separate chains", color="C1")
plt.show()
wh.listen("separate_chains.mid")

The two chains do not know about each other: a pitch can land on any rhythm.

## One chain for both: symbols can be anything

A symbol does not have to be a number. If we glue each pitch to its rhythm, `(pitch, delta)`,
and learn a chain over these pairs, the chain remembers **which rhythm belonged to which pitch**.
Note that the seed is a *list* of pairs (two of them, because the order is 2).

In [ ]:
pairs = list(zip(melody.pitches, melody.deltas))
print(pairs[:4])

joint_chain = MarkovChain()
joint_chain.generateRulesFromList(pairs, order=2)

joint = joint_chain.getNElements(seed=pairs[:2], n=N)
joint_pitches = [p for p, d in joint]
joint_rhythm  = [d for p, d in joint]

In [ ]:
wh.render_midi(joint_pitches, joint_rhythm, melody.bpm, "joint_chain.mid")
wh.plot_notes(joint_pitches, joint_rhythm, title="One chain for pitch and rhythm", color="C2")
plt.show()
wh.listen("joint_chain.mid")

Compare the three versions by ear: borrowed rhythm (chapter 2), separate chains, joint chain.
The joint chain keeps the characteristic note–rhythm combinations of the original; the separate
chains mix them up more freely.

**Try it:**

- Use `order=1` for the joint chain. What changes?
- Learn the joint chain with `order=3`. Does the result still differ from the original?
- Which combination (orders for pitch and rhythm) gives your favourite result?